In [9]:
library(tidyverse)
library(BiocParallel)
library(parallel)
library(sva)
library(dplyr)
library(ggplot2)
library(magrittr)

── Attaching packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.6     ✔ purrr   0.3.4
✔ tibble  3.1.7     ✔ dplyr   1.0.9
✔ tidyr   1.2.0     ✔ stringr 1.4.0
✔ readr   2.1.2     ✔ forcats 0.5.1

── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Loading required package: mgcv

Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:


In [20]:
#command line arguments
args = commandArgs(trailingOnly = TRUE)


In [21]:
#command line arguments
#countspath = args[1]
countspath =  "example-data/rnaseq/counts_bc-in.tsv"
#coldatapath = args[2]
coldatapath = "example-data/rnaseq/coldata_bc-in.tsv"
#user_directory = args[3]


In [22]:
#importing the sample data and counts
counts <- readr::read_tsv(countspath)
metadata <- readr::read_tsv(coldatapath)

Rows: 20242 Columns: 11
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): Hugo_Symbol
dbl (10): GTEX-ZLWG-1026-SM-4WWC4, GTEX-13OVJ-2326-SM-5IJGA, GTEX-14BIM-2226...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10 Columns: 3
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): sample_name, condition
dbl (

In [23]:
#creating correct data structures for batch correction
rma_expr <- as.matrix(counts[-1])
rownames(rma_expr) <- counts$Hugo_Symbol
model_m <- model.matrix(~ condition, data = metadata)
batch <- metadata$batch

In [24]:
#getting label for output
labels <- counts$Hugo_Symbol
label <- data.frame(labels)

In [15]:
rma_expr

,GTEX-ZLWG-1026-SM-4WWC4,GTEX-13OVJ-2326-SM-5IJGA,GTEX-14BIM-2226-SM-5SI8Y,GTEX-Q734-1626-SM-48U1B,GTEX-S7SF-0826-SM-4AD4W,GTEX-12WSD-2826-SM-59HKT,GTEX-145ME-1326-SM-5O98Q,GTEX-S341-1026-SM-4AD71,GTEX-P4PP-2026-SM-3P61N,GTEX-113JC-2226-SM-5EGJG
FAM208A,2715.29,2192.72,3086.56,2888.33,2074.97,3125.08,2388.48,1791.53,3661.58,2231.58
RADIL,517.00,914.00,884.00,361.00,326.00,708.00,235.00,226.00,1614.00,811.00
AP1M2,4.00,2.00,2.00,6.00,19.00,4.00,5.00,23.00,66.00,3.00
TAF1,2673.00,3581.00,2568.00,2741.00,1711.85,3010.35,2272.00,1632.98,5170.20,2064.30
SIGLEC5,9.49,18.69,11.83,17.94,47.18,3.90,9.26,0.00,6.37,9.34
KLF1,1.00,7.00,1.00,0.00,0.00,0.00,2.00,0.00,0.00,1.00
USHBP1,334.00,529.00,920.62,924.00,1120.00,730.00,457.00,705.00,528.00,205.00
LRRC38,2.00,0.00,2.00,0.00,3.00,0.00,0.00,11.00,2.00,3.00
NKPD1,10.00,9.00,4.76,22.40,6.32,10.00,12.00,6.00,23.00,7.12
SLC26A8,8.00,8.00,12.00,4.00,6.00,6.00,2.00,4.00,4.00,17.00


In [25]:
labels

[1] "FAM208A"            "RADIL"              "AP1M2"             
    [4] "TAF1"               "SIGLEC5"            "KLF1"              
    [7] "USHBP1"             "LRRC38"             "NKPD1"             
   [10] "SLC26A8"            "SGCA"               "PGF"               
   [13] "OR5H14"             "KLHL1"              "TUBA3C"            
   [16] "UFM1"               "TNPO2"              "NACA2"             
   [19] "TMEM256"            "IFI35"              "AP001362.1"        
   [22] "KLF16"              "AC026703.1"         "TCF21"             
   [25] "GNL2"               "OR11G2"             "SLC5A7"            
   [28] "OR5A2"              "MTRNR2L1"           "SORCS1"            
   [31] "NMS"                "ELAVL3"             "RAB11FIP4"         
   [34] "UBR5"               "NOTCH2"             "LL22NC03-63E9.3"   
   [37] "LAMTOR2"            "TIMP4"              "CASP16"            
   [40] "8-Sep"              "GGT7"               "APEH"              
   [43] "ASXL1"              "GNG10"              "GORASP1"           
   [46] "SPERT"              "STAU1"              "KCNS1"             
   [49] "YTHDC1"             "GNA12"              "AKR1E2"            
   [52] "LDLR"               "RNF169"             "CFHR2"             
   [55] "KRT9"               "ORMDL2"             "OR51B5"            
   [58] "SPINT4"             "RAC1"               "CIZ1"              
   [61] "LYST"               "SHISA9"             "B4GALT6"           
   [64] "HSFY2"              "AUH"                "NT5DC1"            
   [67] "AP001421.1"         "SERINC4"            "ATP13A2"           
   [70] "CTRL"               "MALSU1"             "HRH4"              
   [73] "P2RY11"             "INF2"               "NEDD4L"            
   [76] "TBCE"               "TMLHE"              "NLRP1"             
   [79] "ZMYND8"             "RP11-345J4.3"       "EIF5AL1"           
   [82] "MCM10"              "SREBF2"             "RPP30"             
   [85] "ACAD10"             "PITRM1"             "RP11-351M8.1"      
   [88] "AC011308.1"         "SLC36A4"            "GEMIN5"            
   [91] "SSC5D"              "NHLH2"              "RHBDL1"            
   [94] "EIF2B2"             "TARDBP"             "EMC7"              
   [97] "DDX58"              "C1QTNF6"            "C8orf59"           
  [100] "NME3"               "TNC"                "NBPF20"            
  [103] "CXorf40B"           "WDR82"              "MYO1E"             
  [106] "STC2"               "POTEE"              "OR9A2"             
  [109] "CDC23"              "SNX27"              "SLC6A3"            
  [112] "GRAP2"              "TMC3"               "TOMM6"             
  [115] "CCT5"               "RGS17"              "YKT6"              
  [118] "RPL36A"             "AC017081.1"         "GGNBP2"            
  [121] "NOL10"              "CTU2"               "BARX1"             
  [124] "NELFCD"             "KBTBD3"             "USP30"             
  [127] "NFAM1"              "ZNF850"             "SUSD2"             
  [130] "UBE2U"              "ARL17B"             "KMO"               
  [133] "ALX1"               "CORO7"              "OR11A1"            
  [136] "CITED2"             "TP53TG3D"           "ARL5B"             
  [139] "NEO1"               "ELF2"               "SDR42E1"           
  [142] "NBPF4"              "KIF13B"             "NOTCH3"            
  [145] "SF3A3"              "OLFM1"              "CETN2"             
  [148] "CDH26"              "FBXO30"             "MIPOL1"            
  [151] "OR5AK2"             "SLC48A1"            "FAM19A5"           
  [154] "LY6K"               "OR2T10"             "TLR8"              
  [157] "EIF5"               "AP2S1"              "MAGEL2"            
  [160] "RP11-383H13.1"      "ATIC"               "C9orf147"          
  [163] "RP11-178L8.4"       "REXO1L1"            "CEP68"             
  [166] "ELMSAN1"            "IL10RA"             "EZH1"              
  [169] "SHARPIN"           

In [26]:
label

labels
<chr>
FAM208A
RADIL
AP1M2
TAF1
SIGLEC5
KLF1
USHBP1
LRRC38
NKPD1


In [27]:
model_m

,(Intercept),conditionhealthy
1,1,1
2,1,1
3,1,1
4,1,1
5,1,1
6,1,1
7,1,1
8,1,0
9,1,1
10,1,0


In [30]:
batch

[1] 1 1 1 1 1 2 2 2 2 2

In [16]:
#batch correction!
#bc_rma_expr <- ComBat_seq(rma_expr, batch = batch, mod = model_m, ref.batch = 1)

bc_rma_expr <- ComBat_seq(rma_expr, batch = batch, group = NULL, covar_mod=model_m)

#bc_rma_expr <- ComBat_seq(rma_expr <- rma_expr[,colnames(rma_expr)!="Hugo_Symbol"], batch = batch)

Found 2 batches
Using null model in ComBat-seq.
Adjusting for 1 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data


In [17]:
#editing and combining files
final <- data.frame(bc_rma_expr)
final2 <- bind_cols(label,final)

In [19]:
#writing out
write_tsv(final2,"bc_counts.tsv")